# Chọn trọng số ensemble C30 bằng OOF

Người thực hiện: Huy. Vai trò: thí nghiệm bổ sung về ensemble.
Nguồn mô hình thành phần: bộ Claude tái dựng trong Params1, không tuning lại tham số.


## 1. Nhắc lại cấu hình thí nghiệm trước
Thí nghiệm kiểm tra liệu thay trọng số kết hợp Random Forest và XGBoost có giảm MAE so với ensemble 50/50 hay không. Giữ nguyên C30, bộ tham số Claude, cohort và ba fold của FE. Chỉ thay trọng số ensemble; không tuning lại mô hình thành phần. Trọng số được chọn bằng MAE trung bình ba fold trên train, không sử dụng validation hoặc test để lựa chọn.

Chọn trọng số RF–XGBoost bằng OOF, giữ ensemble 50/50

$
\hat y = w\hat y_{RF}+(1-w)\hat y_{XGB},\quad
w\in\{0;0,1;\ldots;1\}
$
- w = 0: chỉ dùng XGBoost.
- w = 0.5: đối chứng 50/50.
- w = 1: chỉ dùng RF.

## 2. Đọc giao thức feature engineering và tham số Claude

Đọc bảng fold để thấy số mẫu và khoảng thời gian. Không gọi ba fold là ba khoảng lịch bằng nhau.

In [1]:
from pathlib import Path
import sys
import json
import hashlib
from importlib.metadata import version

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "data/preprocessed/hourly_unfilled.csv").is_file()
)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src import feature_experiments as fe
from src.features import CONFIG_C
from src.model import make_cv_model
from src.evaluation import regression_metrics

source_path = (
    project_root
    / "experiments/20-11-2026_Huy_C30_Params1/outputs/run_config.json"
)

source_config = json.loads(source_path.read_text(encoding="utf-8"))
model_params = source_config["explicit_params"]

context = fe.load_protocol(project_root)

assert list(source_config["features"]) == list(CONFIG_C)
assert len(context["cohort"]) == source_config["n_train"]
assert fe.frame_hash(
    context["cohort"][["date_time", "traffic_volume"]]
) == source_config["train_cohort_hash"], "Cohort train khác lượt fit Claude."

source_hashes = {
    filename: hashlib.sha256((project_root / filename).read_bytes()).hexdigest()
    for filename in source_config["source_hashes"]
}
assert source_hashes == source_config["source_hashes"], (
    "Code tạo feature, tiền xử lý hoặc mô hình đã đổi; cần kiểm tra trước khi dùng lại validation."
)
versions = {
    "numpy": version("numpy"), "pandas": version("pandas"),
    "sklearn": version("scikit-learn"), "xgboost": version("xgboost"),
}
assert all(versions[name] == source_config["versions"][name] for name in versions), (
    "Phiên bản thư viện khác lượt fit validation được dùng lại."
)

display(context["manifest"])
display(pd.DataFrame([
    {"model": name, "parameters": str(parameters)}
    for name, parameters in model_params.items()
]))

,fold,fit_start,fit_end,score_start,score_end,n_fit,n,fit_calendar_hours,score_calendar_hours,fit_hash,score_hash
0,1,2012-10-09 09:00:00,2013-08-05 19:00:00,2013-08-05 20:00:00,2015-07-12 06:00:00,5628,5625,7211.0,16931.0,907f339dcf7453df3f5e70f83fd477f72bdfd531e01eae...,3e445aa0bc2af81e32b3d2126a6f9e34073d17a6872bc0...
1,2,2012-10-09 09:00:00,2015-07-12 06:00:00,2015-07-12 07:00:00,2016-09-07 17:00:00,11253,5625,24142.0,10163.0,0a956b96e0a294dfd75ca7d8b49aa36b9e6cbffd755794...,a1d7f7f4f11fc1da3571b8dfdba463987d69e3ef1adee3...
2,3,2012-10-09 09:00:00,2016-09-07 17:00:00,2016-09-07 18:00:00,2017-05-10 04:00:00,16878,5625,34305.0,5867.0,a64f6fc3241daadbcf0a479c22878e11e10c69bfa3c4e2...,4117d3a132e44ecc9c0f8eea01e43dc911e7758ea92dc3...


,model,parameters
0,Random Forest,"{'model__n_estimators': 100, 'model__min_sampl..."
1,XGBoost,"{'model__n_estimators': 400, 'model__max_depth..."


## 3. Tạo OOF cho hai mô hình

Trong mỗi fold, thống kê điền thiếu chỉ được học từ fold-train. Hai mô hình dùng cùng dữ liệu huấn luyện và cùng giờ chấm. Mỗi mô hình fit một lần ở mỗi fold, sau đó lưu dự đoán để thử các trọng số mà không phải huấn luyện lại.

In [2]:
oof_parts = []
preprocessing_logs = []

for fold_number, (fit_indices, score_indices) in enumerate(
    context["folds"], start=1
):
    fit_rows = context["cohort"].iloc[fit_indices]
    score_rows = context["cohort"].iloc[score_indices]

    X_fit, X_score, statistics = fe.prepare_fold(
        context["train"],
        context["featured"],
        fit_rows["date_time"],
        score_rows["date_time"],
    )

    assert X_score.index.equals(
        pd.DatetimeIndex(score_rows["date_time"])
    )

    fold_predictions = pd.DataFrame({
        "fold": fold_number,
        "date_time": score_rows["date_time"].to_numpy(),
        "y_true": score_rows["traffic_volume"].to_numpy(),
    })

    for model_name in ("Random Forest", "XGBoost"):
        model = make_cv_model(model_name, CONFIG_C)
        model.set_params(**model_params[model_name])

        model.fit(X_fit, fit_rows["traffic_volume"].to_numpy())

        fold_predictions[model_name] = np.clip(
            model.predict(X_score), 0, None
        )

    oof_parts.append(fold_predictions)
    preprocessing_logs.append({
        "fold": fold_number,
        **statistics,
    })

    print(f"Đã hoàn thành fold {fold_number}")

oof = (
    pd.concat(oof_parts, ignore_index=True)
    .sort_values("date_time")
    .reset_index(drop=True)
)

assert oof["date_time"].is_unique
assert np.isfinite(
    oof[["y_true", "Random Forest", "XGBoost"]].to_numpy()
).all()

display(oof.head())
print("Số giờ có dự đoán OOF:", len(oof))

Đã hoàn thành fold 1
Đã hoàn thành fold 2
Đã hoàn thành fold 3
Số giờ có dự đoán OOF: 16875


,fold,date_time,y_true,Random Forest,XGBoost
0,1,2013-08-05 20:00:00,2834,2981.089607,2963.206543
1,1,2013-08-05 21:00:00,2516,2664.788170,2681.653076
2,1,2013-08-05 22:00:00,1818,1810.226435,1754.257080
3,1,2013-08-05 23:00:00,1179,1258.469136,1280.272705
4,1,2013-08-06 01:00:00,410,427.204536,363.401550


Với giao thức hiện tại, dự kiến có 16.875 giờ OOF. Phần đầu cohort chỉ làm train nên không có OOF.
## 4. Chấm 11 trọng số

Dùng cùng các dự đoán OOF để đánh giá mọi trọng số. Báo MAE, RMSE, R² theo từng fold. Tiêu chí chọn là MAE trung bình fold; nếu bằng nhau, ưu tiên trọng số gần 50/50, sau đó mới xét RMSE.

In [3]:
weights = np.round(np.linspace(0, 1, 11), 1)
score_rows = []

for weight_rf in weights:
    for fold_number, frame in oof.groupby("fold"):
        prediction = (
            weight_rf * frame["Random Forest"]
            + (1 - weight_rf) * frame["XGBoost"]
        )

        score_rows.append({
            "weight_rf": weight_rf,
            "weight_xgb": 1 - weight_rf,
            "fold": fold_number,
            "n": len(frame),
            **regression_metrics(frame["y_true"], prediction),
        })

weight_scores = pd.DataFrame(score_rows)

weight_summary = (
    weight_scores.groupby("weight_rf", as_index=False)
    .agg(
        n=("n", "sum"),
        MAE_mean=("MAE", "mean"),
        MAE_std=("MAE", "std"),
        RMSE_mean=("RMSE", "mean"),
        R2_mean=("R2", "mean"),
    )
)

weight_summary["distance_50"] = (
    weight_summary["weight_rf"] - 0.5
).abs()

weight_summary = weight_summary.sort_values(
    ["MAE_mean", "distance_50", "RMSE_mean"]
).reset_index(drop=True)

best_weight = float(weight_summary.iloc[0]["weight_rf"])

display(weight_summary.round(4))
print("Trọng số RF được chọn:", best_weight)
print("Trọng số XGBoost:", 1 - best_weight)

Trọng số RF được chọn: 0.6
Trọng số XGBoost: 0.4


,weight_rf,n,MAE_mean,MAE_std,RMSE_mean,R2_mean,distance_50
0,0.6,16875,227.2161,12.9997,372.9077,0.9637,0.1
1,0.5,16875,227.2410,13.0267,372.7650,0.9637,0.0
2,0.7,16875,227.4742,13.0013,373.4483,0.9636,0.2
3,0.4,16875,227.5205,13.0355,373.0210,0.9637,0.1
4,0.8,16875,228.0002,12.9981,374.3852,0.9634,0.3
5,0.3,16875,228.0744,12.9964,373.6746,0.9635,0.2
6,0.9,16875,228.8026,13.0276,375.7151,0.9631,0.4
7,0.2,16875,228.9213,12.9183,374.7239,0.9633,0.3
8,1.0,16875,229.8607,13.0563,377.4338,0.9628,0.5
9,0.1,16875,230.0659,12.8193,376.1653,0.9630,0.4


## 5. Đối chiếu với 50/50 theo từng fold

Kiểm tra lợi ích có xuất hiện ở nhiều fold hay chỉ tập trung ở một giai đoạn. Quy ước chênh lệch MAE dương nghĩa trọng số được chọn có sai số thấp hơn đối chứng 50/50.

In [4]:
baseline = (
    weight_scores[weight_scores["weight_rf"] == 0.5]
    .set_index("fold")
)

selected = (
    weight_scores[weight_scores["weight_rf"] == best_weight]
    .set_index("fold")
)

fold_comparison = pd.DataFrame({
    "n": selected["n"],
    "MAE_50_50": baseline["MAE"],
    "MAE_selected": selected["MAE"],
    "MAE_reduction": baseline["MAE"] - selected["MAE"],
    "RMSE_50_50": baseline["RMSE"],
    "RMSE_selected": selected["RMSE"],
    "RMSE_reduction": baseline["RMSE"] - selected["RMSE"],
}).reset_index()

display(fold_comparison.round(3))

,fold,n,MAE_50_50,MAE_selected,MAE_reduction,RMSE_50_50,RMSE_selected,RMSE_reduction
0,1,5625,219.409,219.441,-0.032,367.851,368.143,-0.292
1,2,5625,242.279,242.224,0.055,413.189,413.390,-0.200
2,3,5625,220.036,219.984,0.052,337.255,337.191,0.064


OOF dùng để lựa chọn trọng số nên điểm tại trọng số được chọn không phải đánh giá độc lập. Báo chênh lệch theo từng fold; mức giảm nhỏ hoặc trái chiều giữa các fold chưa cho thấy lợi ích ổn định. Không đổi lưới trọng số sau khi xem validation.

## 6. Cố định trọng số và lưu bằng chứng OOF

Lưu kết quả trong outputs/ cùng thư mục notebook. Ghi cả hash cohort, các fold, tham số, phiên bản thư viện và hash code. Phần validation bên dưới chỉ đọc trọng số đã lưu, không tìm lại trọng số. OOF và validation là hai giai đoạn khác nhau, không đối chiếu trực tiếp MAE của chúng để kết luận cải thiện.

In [5]:
output_dir = project_root / "experiments/21-11-2026_Huy_C30_OOF/outputs"
output_dir.mkdir(parents=True, exist_ok=True)

oof_export = oof.copy()
oof_export[["Random Forest", "XGBoost"]] = oof_export[
    ["Random Forest", "XGBoost"]
].astype("float64")
oof_export.to_csv(output_dir / "oof_predictions.csv", index=False, float_format="%.17g")
weight_scores.to_csv(output_dir / "weight_fold_scores.csv", index=False)
weight_summary.to_csv(output_dir / "weight_summary.csv", index=False)
fold_comparison.to_csv(output_dir / "comparison_50_50.csv", index=False)
context["manifest"].to_csv(output_dir / "fold_manifest.csv", index=False)

selection = {
    "config": "C30",
    "features": list(CONFIG_C),
    "model_params": model_params,
    "parameter_source": str(source_path.relative_to(project_root)),
    "weight_rf": best_weight,
    "weight_xgb": 1 - best_weight,
    "candidate_weights": weights.tolist(),
    "selection_rule": "mean fold MAE; then nearest 0.5; then mean fold RMSE",
    "cohort_hash": fe.frame_hash(
        context["cohort"][["date_time", "traffic_volume"]]
    ),
    "preprocessing": preprocessing_logs,
    "validation_used_for_weight_selection": False,
    "validation_previously_seen": True,
    "source_hashes": source_hashes,
    "versions": versions,
    "folds": context["manifest"].to_dict("records"),
    "parameter_source_sha256": hashlib.sha256(source_path.read_bytes()).hexdigest(),
}

(output_dir / "selected_weights.json").write_text(
    json.dumps(selection, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)

Out[0]: 83161


## 7. Đọc validation sau khi đã cố định trọng số

Dùng lại dự đoán của hai mô hình đã fit trên toàn bộ cohort train trong Params1. Kiểm tra tham số, hash code, phiên bản thư viện, timestamp và target trước khi dùng lại. Không fit hoặc chọn trọng số trên validation; test không được sử dụng. Validation đã được xem ở các lượt trước nên đây là đối chiếu bổ sung.

In [6]:
selected_weights = json.loads(
    (output_dir / "selected_weights.json").read_text(encoding="utf-8")
)
assert selected_weights["model_params"] == source_config["explicit_params"]
assert selected_weights["features"] == source_config["features"]
assert selected_weights["cohort_hash"] == source_config["train_cohort_hash"]

validation_folder = source_path.parent
validation = pd.read_csv(
    validation_folder / "validation_predictions.csv",
    parse_dates=["date_time"],
    float_precision="round_trip",
)
# Khôi phục dự đoán XGBoost ở float32 gốc từ CSV, rồi tính ensemble ở float64.
validation["XGBoost"] = validation["XGBoost"].astype("float32").astype("float64")
assert validation["date_time"].is_unique
assert validation["date_time"].is_monotonic_increasing
assert len(validation) == source_config["n_validation"]
validation_cohort = validation[["date_time", "y_true"]].rename(
    columns={"y_true": "traffic_volume"}
)
assert fe.frame_hash(validation_cohort) == source_config["validation_cohort_hash"]
assert np.isfinite(validation[["y_true", "Random Forest", "XGBoost"]].to_numpy()).all()
assert context["cohort"]["date_time"].max() < validation["date_time"].min()

display(pd.DataFrame([{
    "split": "validation", "n": len(validation),
    "start": validation["date_time"].min(), "end": validation["date_time"].max(),
    "weight_rf": selected_weights["weight_rf"],
    "weight_xgb": selected_weights["weight_xgb"],
}]))

,split,n,start,end,weight_rf,weight_xgb
0,validation,6050,2017-05-10 05:00:00,2018-01-19 14:00:00,0.6,0.4


## 8. Chấm cùng các giờ validation

Đối chiếu RF, XGBoost, ensemble 50/50 và ensemble dùng trọng số chọn bằng OOF. Kiểm tra điểm 50/50 đọc lại khớp file nguồn trước khi chấm phương án mới. Chênh lệch dương trong bảng sau nghĩa phương án OOF giảm sai số so với 50/50.

In [7]:
weight_rf = float(selected_weights["weight_rf"])
weight_xgb = float(selected_weights["weight_xgb"])
assert 0 <= weight_rf <= 1 and np.isclose(weight_rf + weight_xgb, 1)
weighted_prediction = (
    weight_rf * validation["Random Forest"]
    + weight_xgb * validation["XGBoost"]
)
assert np.allclose(
    validation["Ensemble RF + XGBoost (50/50)"],
    0.5 * validation["Random Forest"] + 0.5 * validation["XGBoost"],
    rtol=0, atol=1e-8,
)
saved_metrics = pd.read_csv(validation_folder / "validation_metrics.csv").set_index("model")
validation_rows = []
for model_name, prediction in {
    "Random Forest": validation["Random Forest"],
    "XGBoost": validation["XGBoost"],
    "Ensemble 50/50": validation["Ensemble RF + XGBoost (50/50)"],
    "Ensemble chọn bằng OOF": weighted_prediction,
}.items():
    scores = regression_metrics(validation["y_true"], prediction)
    original_name = "Ensemble RF + XGBoost (50/50)" if model_name == "Ensemble 50/50" else model_name
    if original_name in saved_metrics.index:
        for metric, value in scores.items():
            assert np.isclose(value, saved_metrics.loc[original_name, metric], rtol=0, atol=1e-6)
    validation_rows.append({"model": model_name, "n": len(validation), **scores})

validation_metrics = pd.DataFrame(validation_rows)
display(validation_metrics.sort_values("MAE").round({"MAE": 3, "RMSE": 3, "R2": 6}))
indexed_metrics = validation_metrics.set_index("model")
validation_delta = pd.DataFrame([{
    "comparison": "50/50 trừ ensemble OOF", "n": len(validation),
    "MAE_reduction": indexed_metrics.loc["Ensemble 50/50", "MAE"] - indexed_metrics.loc["Ensemble chọn bằng OOF", "MAE"],
    "RMSE_reduction": indexed_metrics.loc["Ensemble 50/50", "RMSE"] - indexed_metrics.loc["Ensemble chọn bằng OOF", "RMSE"],
}])
display(validation_delta.round(4))
validation_metrics.to_csv(output_dir / "validation_metrics.csv", index=False)
validation_delta.to_csv(output_dir / "validation_deltas.csv", index=False)
validation.assign(weighted_ensemble=weighted_prediction).to_csv(
    output_dir / "validation_predictions.csv", index=False, float_format="%.17g"
)

,model,n,MAE,RMSE,R2
2,Ensemble 50/50,6050,193.810,316.085,0.974109
3,Ensemble chọn bằng OOF,6050,193.933,317.580,0.973864
0,Random Forest,6050,196.775,326.773,0.972329
1,XGBoost,6050,197.031,313.640,0.974508


,comparison,n,MAE_reduction,RMSE_reduction
0,50/50 trừ ensemble OOF,6050,-0.1223,-1.4955


## 9. Tổng kết thí nghiệm

Tổng kết dùng số tính từ bảng kết quả. Phân biệt trọng số được chọn bằng OOF với quyết định giữ hoặc thay đối chứng sau đánh giá validation. Không dùng validation để dò lại trọng số; chưa kết luận từ test.

In [8]:
oof_baseline = weight_summary.loc[np.isclose(weight_summary["weight_rf"], 0.5)].iloc[0]
oof_selected = weight_summary.loc[np.isclose(weight_summary["weight_rf"], weight_rf)].iloc[0]
oof_gain = float(oof_baseline["MAE_mean"] - oof_selected["MAE_mean"])
mae_gain = float(validation_delta.loc[0, "MAE_reduction"])
rmse_gain = float(validation_delta.loc[0, "RMSE_reduction"])
if mae_gain > 0:
    decision = "Phương án chọn bằng OOF giảm MAE validation trong lượt này; cần cân nhắc mức giảm và đánh đổi RMSE trước khi chốt."
else:
    decision = "Phương án chọn bằng OOF chưa cải thiện MAE validation; giữ ensemble 50/50 làm phương án hiện tại."
assessment = (
    f"OOF chọn RF {weight_rf:.0%} và XGBoost {weight_xgb:.0%}. "
    f"MAE trung bình fold đổi từ {oof_baseline['MAE_mean']:.4f} thành {oof_selected['MAE_mean']:.4f} xe/giờ, "
    f"giảm {oof_gain:.4f} xe/giờ. "
    f"Trên {len(validation):,} giờ validation, chênh MAE (50/50 trừ OOF) là {mae_gain:.4f} "
    f"và chênh RMSE là {rmse_gain:.4f} xe/giờ. {decision} "
    "Validation đã được xem nhiều lần; đây là thí nghiệm bổ sung, chưa đánh giá test."
)
display(Markdown(assessment))
(output_dir / "assessment.md").write_text(assessment + "\n", encoding="utf-8")

Out[0]: 387


OOF chọn RF 60% và XGBoost 40%. MAE trung bình fold đổi từ 227.2410 thành 227.2161 xe/giờ, giảm 0.0249 xe/giờ. Trên 6,050 giờ validation, chênh MAE (50/50 trừ OOF) là -0.1223 và chênh RMSE là -1.4955 xe/giờ. Phương án chọn bằng OOF chưa cải thiện MAE validation; giữ ensemble 50/50 làm phương án hiện tại. Validation đã được xem nhiều lần; đây là thí nghiệm bổ sung, chưa đánh giá test.